In [ ]:
# =============================================================================
# Fig5 (restructured):
#   a: The original quadrant scatter plots (Δ Occ_before vs Δ Occ_after) for
#      three periods: 2020→2040, 2040→2060, 2020→2060.
#   b: Mean bed occupancy rate 2020-2060, before (blue) vs after (red) mobility,
#      one small subplot per income quintile, x-axis = year.
#   c: Bar plots of the relative change (percentage change) in occupancy over time,
#      one subplot per income quintile, showing trend across years.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ── 0. Paths & settings (same sources as Fig4) ────────────────────────────────
CITYSUM_BYDISEASE_DIR = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease")
CITYSUM_BYDISEASE_TPL = "citysum_bydisease_{air}_{year}.csv"

DISEASE_FLOW_DIR = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results")
DISEASE_FLOW_TPL = "flow_patientnum_{disease_tag}_{air}_{year}.csv"

INPUT_FILE = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/air_pollution/data source/hospital/13-National hospital directory.xlsx")
INCOME_DIR       = Path("/Volumes/UCL/论文工作/空气污染/weighted_gdp/avg_fer_rcp/avg_2020.csv")
INCOME_CITY_COL  = "city"
INCOME_CLASS_COL = "income_group"

SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")

OUTFILE = Path("/Users/shirley/Desktop/plots_V2/Fig5_trend_violin_quadrant.png")
OUTFILE.parent.mkdir(parents=True, exist_ok=True)

SCENARIO = "earlypeak_NZ_CL"

GDP_CLASS_ORDER  = ["Low", "Lower middle", "Middle", "Upper middle", "High"]
GDP_CLASS_SHORT  = {"Low": "Low", "Lower middle": "L-Mid", "Middle": "Mid",
                     "Upper middle": "U-Mid", "High": "High"}
GDP_CLASS_COLORS = {"Low": "#D6EFD6", "Lower middle": "#A8DDA8", "Middle": "#6FC26F",
                     "Upper middle": "#3B9E3B", "High": "#1B6B1B"}

TREND_YEARS = [2020, 2030, 2040, 2050, 2060]

PERIODS = [
    (2020, 2040, "2020\u20132040"),
    (2040, 2060, "2040\u20132060"),
    (2020, 2060, "2020\u20132060"),
]

STATE_COLORS = {"before": "#4e79a7", "after": "#e15759"}

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        7,
    "axes.titlesize":   8,
    "axes.titleweight": "bold",
    "axes.titlepad":    3,
})

# ── 0b. Length-of-stay lookup ──────────────────────────────────────────────────
LOS_BY_SHORT_NAME = {
    "COPD": 10.0, "IHD": 12.6, "LRI": 11.0, "LC": 14.2, "STROKE": 19.7,
}

def match_los(disease_name: str) -> float:
    s = disease_name.strip().lower()
    if "copd" in s or "chronic obstructive" in s:
        return LOS_BY_SHORT_NAME["COPD"]
    if "isch" in s or "ihd" in s or "coronary" in s or "chd" in s:
        return LOS_BY_SHORT_NAME["IHD"]
    if "lower respiratory" in s or s == "lri" or "lri" in s:
        return LOS_BY_SHORT_NAME["LRI"]
    if "lung cancer" in s or s == "lc":
        return LOS_BY_SHORT_NAME["LC"]
    if "stroke" in s:
        return LOS_BY_SHORT_NAME["STROKE"]
    raise ValueError(f"Could not match disease name '{disease_name}' to a known LOS category.")

CITY_NAME_MAP = {
    "Wulumuqi": "Urumqi", "Xian": "Xi'an", "Qiqihaer": "Qiqihar",
    "Huhehaote": "Hohhot", "Haerbin": "Harbin", "Xiuqian": "Suqian",
    "Wulanchabu": "Ulanqab", "Shaoang": "Zhaotong", "Tongcuan": "Tongchuan",
    "Xiangfan": "Xiangyang", "Akesu": "Aksu", "Alaer": "Alar",
    "Alashan": "Alxa", "Bayanzhuoer": "Bayannur", "Bayinguoleng": "Bayingolin",
    "Boertalameng": "Bortala", "Changdu": "Qamdo", "Eerduosi": "Erdos",
    "Ordos": "Erdos", "Guoluo": "Golog", "Guolo": "Golog",
    "Hulunbeier": "Hulunbuir", "Kezilesu": "Kizilsu", "Ledong": "Ledong",
    "Lingshui": "Lingshui", "Linzhi": "Nyingchi", "Naqu": "Nagqu",
    "Qiongzhong": "Qiongzhong", "Shennongjia": "Shennongjia",
    "Tumushuke": "Tumxuk", "Xilinguole": "Xilingol", "Xingan": "Hinggan",
}

def disease_to_tag(disease):
    return disease.strip().replace(" ", "_").replace("/", "-")

# ── 1. Per-disease occupancy loaders ──────────────────────────────────────────
def load_local_bydisease(year):
    path = CITYSUM_BYDISEASE_DIR / CITYSUM_BYDISEASE_TPL.format(air=SCENARIO, year=year)
    df = pd.read_csv(path)
    df["city"]    = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df["disease"] = df["disease"].astype(str).str.strip()
    return df

def load_disease_flow_matrix(year, disease_tag):
    path = DISEASE_FLOW_DIR / DISEASE_FLOW_TPL.format(disease_tag=disease_tag, air=SCENARIO, year=year)
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0)
    df.index   = df.index.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df.columns = df.columns.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    return df

def compute_year_occupancy_days(year):
    local_df = load_local_bydisease(year)
    diseases = sorted(local_df["disease"].unique().tolist())

    before_acc, after_acc = None, None
    for disease in diseases:
        los = match_los(disease)
        disease_tag = disease_to_tag(disease)

        sub = local_df[local_df["disease"] == disease].groupby("city").sum(numeric_only=True)
        mo_total_d = sub["mo_total_sum"]
        local_mo_d = sub["local_mo_total"]

        flow_mat = load_disease_flow_matrix(year, disease_tag)
        if flow_mat is None:
            inflow_d = pd.Series(0.0, index=mo_total_d.index)
        else:
            inflow_d = flow_mat.sum(axis=0).groupby(level=0).sum()

        all_cities = mo_total_d.index.union(local_mo_d.index).union(inflow_d.index)
        mo_total_d = mo_total_d.reindex(all_cities, fill_value=0.0)
        local_mo_d = local_mo_d.reindex(all_cities, fill_value=0.0)
        inflow_d   = inflow_d.reindex(all_cities, fill_value=0.0)

        demand_after_d = (local_mo_d + inflow_d).clip(lower=0.0)

        before_days_d = mo_total_d * los
        after_days_d  = demand_after_d * los

        before_acc = before_days_d if before_acc is None else before_acc.add(before_days_d, fill_value=0.0)
        after_acc  = after_days_d  if after_acc  is None else after_acc.add(after_days_d,  fill_value=0.0)

    return pd.DataFrame({"occ_before_days": before_acc, "occ_after_days": after_acc})

# ── 2. Resource & income-class loaders ────────────────────────────────────────
def load_resource():
    df = pd.read_excel(INPUT_FILE, sheet_name="fig4")
    df = df[["city", "beds", "clinics"]].copy()
    df["beds"]    = pd.to_numeric(df["beds"],    errors="coerce")
    df["clinics"] = pd.to_numeric(df["clinics"], errors="coerce")
    df = df.dropna(subset=["city", "beds", "clinics"])
    df = df[(df["beds"] > 0) & (df["clinics"] > 0)]
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")["beds"].sum()

def load_gdp_class():
    df = pd.read_csv(INCOME_DIR)
    df[INCOME_CITY_COL] = df[INCOME_CITY_COL].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df = df.dropna(subset=[INCOME_CITY_COL, INCOME_CLASS_COL])
    df["gdp_class"] = pd.Categorical(df[INCOME_CLASS_COL], categories=GDP_CLASS_ORDER, ordered=True)
    return df.set_index(INCOME_CITY_COL)["gdp_class"]

# ── 3. Compute occupancy (%) for every trend year ─────────────────────────────
resource  = load_resource()
gdp_class = load_gdp_class()

occ_by_year = {}
for year in TREND_YEARS:
    occ_days = compute_year_occupancy_days(year)
    common = occ_days.index.intersection(resource.index)
    occ_days = occ_days.loc[common]
    res = resource.loc[common]
    df_year = pd.DataFrame({
        "Occ_before": occ_days["occ_before_days"] / (res * 365) * 100,
        "Occ_after":  occ_days["occ_after_days"]  / (res * 365) * 100,
    })
    df_year["gdp_class"] = df_year.index.map(gdp_class)
    df_year = df_year.dropna(subset=["gdp_class"])
    occ_by_year[year] = df_year
    print(f"\u2713 Computed occupancy for year {year} | {len(df_year)} cities")

# ── 4. Panel b data: mean Occ_before / Occ_after per income group, per year ──
trend_rows = []
for year in TREND_YEARS:
    df_year = occ_by_year[year]
    grp = df_year.groupby("gdp_class", observed=True)[["Occ_before", "Occ_after"]].mean()
    for cls in GDP_CLASS_ORDER:
        if cls in grp.index:
            trend_rows.append({
                "year": year, "gdp_class": cls,
                "Occ_before": grp.loc[cls, "Occ_before"],
                "Occ_after":  grp.loc[cls, "Occ_after"],
            })
trend_df = pd.DataFrame(trend_rows)

# ── 5. Panel c data: relative change using aggregated data ──────────────────
# IMPORTANT: Use the mean values from panel b to calculate relative change
# This ensures consistency between panel b and panel c
rel_change_agg = []
for year in TREND_YEARS:
    for cls in GDP_CLASS_ORDER:
        sub = trend_df[(trend_df["year"] == year) & (trend_df["gdp_class"] == cls)]
        if len(sub) > 0:
            before = sub["Occ_before"].values[0]
            after = sub["Occ_after"].values[0]
            if before > 0:
                rel_change = (after - before) / before * 100
            else:
                rel_change = np.nan
            rel_change_agg.append({
                "year": year,
                "gdp_class": cls,
                "relative_change": rel_change,
                "Occ_before": before,
                "Occ_after": after,
                "delta_abs": after - before
            })
rel_change_agg_df = pd.DataFrame(rel_change_agg)

# Debug: print out the relative change data using aggregated values
print("\n--- Relative change calculated from aggregated data (panel b means) ---")
for cls in GDP_CLASS_ORDER:
    sub = rel_change_agg_df[rel_change_agg_df["gdp_class"] == cls]
    print(f"\n{cls}:")
    for _, row in sub.iterrows():
        print(f"  {row['year']}: before={row['Occ_before']:.2f}%, after={row['Occ_after']:.2f}%, "
              f"delta={row['delta_abs']:.4f} pp, rel_change={row['relative_change']:.2f}%")
print("-----------------------------------------------------\n")

# ── 6. Panel a data: Δbefore vs Δafter per period ─────────────────────────────
period_data = {}
for y0, y1, label in PERIODS:
    df0, df1 = occ_by_year[y0], occ_by_year[y1]
    common = df0.index.intersection(df1.index)
    d = pd.DataFrame({
        "d_before": df1.loc[common, "Occ_before"] - df0.loc[common, "Occ_before"],
        "d_after":  df1.loc[common, "Occ_after"]  - df0.loc[common, "Occ_after"],
    })
    d["gdp_class"] = d.index.map(gdp_class)
    d = d.dropna(subset=["gdp_class"])
    period_data[label] = d

# ---- 7. Figure layout ----
fig = plt.figure(figsize=(200 / 25.4, 185 / 25.4), dpi=300)
gs = GridSpec(3, 30, figure=fig, hspace=0.40, left=0.08, right=0.98, top=0.95, bottom=0.05,
              height_ratios=[1.0, 0.85, 0.425])

A_COLS = 30 // len(period_data)
B_COLS = 30 // len(GDP_CLASS_ORDER)
C_COLS = 30 // len(GDP_CLASS_ORDER)

# ---- Panel a: quadrant scatter plots ──────────────────────────────────────────
row_a_axes = []

for j, (label, d) in enumerate(period_data.items()):
    ax = fig.add_subplot(gs[0, j*A_COLS:(j+1)*A_COLS])
    row_a_axes.append(ax)

    panel_vals = d[["d_before", "d_after"]].values.flatten()
    panel_vals = panel_vals[np.isfinite(panel_vals)]
    p99 = np.nanpercentile(np.abs(panel_vals), 99) if len(panel_vals) else 1
    true_max = np.nanmax(np.abs(panel_vals)) if len(panel_vals) else 1
    lim = max(p99 * 1.25, true_max * 1.02)

    ax.axhline(0, color="#999999", lw=0.7, zorder=1)
    ax.axvline(0, color="#999999", lw=0.7, zorder=1)

    ax.axhspan(0, lim, xmin=0.5, xmax=1.0, color="#f5f5f5", alpha=0.5, zorder=0)
    ax.axhspan(-lim, 0, xmin=0.5, xmax=1.0, color="#fdecea", alpha=0.6, zorder=0)
    ax.axhspan(-lim, 0, xmin=0.0, xmax=0.5, color="#f5f5f5", alpha=0.5, zorder=0)
    ax.axhspan(0, lim, xmin=0.0, xmax=0.5, color="#eaf5ea", alpha=0.6, zorder=0)

    for cls in GDP_CLASS_ORDER:
        sub = d[d["gdp_class"] == cls]
        ax.scatter(sub["d_after"], sub["d_before"], s=10, color=GDP_CLASS_COLORS[cls],
                   edgecolors="none", alpha=0.85, zorder=3, label=GDP_CLASS_SHORT[cls])

    ax.plot([-lim, lim], [-lim, lim], color="#888888", lw=0.6, ls=(0, (3, 2)), zorder=2)

    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True, symmetric=True))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True, symmetric=True))
    ax.set_xlabel("Bed occupancy rate change, after mobility (pp)", fontsize=6.5)
    if j == 0:
        ax.set_ylabel("Bed occupancy rate change, before mobility (pp)", fontsize=6.5)
    ax.tick_params(labelsize=6)
    ax.set_title(label, fontsize=8, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

    n = len(d)
    q_relief   = ((d["d_after"] < 0) & (d["d_before"] > 0)).sum()
    q_worsen   = ((d["d_after"] > 0) & (d["d_before"] < 0)).sum()
    q_rose     = ((d["d_after"] > 0) & (d["d_before"] > 0)).sum()
    q_fell     = ((d["d_after"] < 0) & (d["d_before"] < 0)).sum()

    def _pct(x, n=n):
        return f"{x/n*100:.0f}%" if n else "0%"

    summary_text = (
        f"n={n}\n"
        f"relief: {q_relief} ({_pct(q_relief)})\n"
        f"worsened: {q_worsen} ({_pct(q_worsen)})\n"
        f"rose regardless: {q_rose} ({_pct(q_rose)})\n"
        f"fell regardless: {q_fell} ({_pct(q_fell)})"
    )
    ax.text(0.97, 0.03, summary_text,
            transform=ax.transAxes, fontsize=5, color="#333333", ha="right", va="bottom",
            linespacing=1.4)

# ---- Panel b: trend lines ─────────────────────────────────────────────────────
a_ymax = max(trend_df["Occ_before"].max(), trend_df["Occ_after"].max()) * 1.15
row_b_axes = []

for j, cls in enumerate(GDP_CLASS_ORDER):
    ax = fig.add_subplot(gs[1, j*B_COLS:(j+1)*B_COLS])
    row_b_axes.append(ax)
    sub = trend_df[trend_df["gdp_class"] == cls].sort_values("year")
    ax.plot(sub["year"], sub["Occ_before"], marker="o", ms=3.5, lw=1.2,
            color=STATE_COLORS["before"], label="Before mobility")
    ax.plot(sub["year"], sub["Occ_after"], marker="o", ms=3.5, lw=1.2,
            color=STATE_COLORS["after"], label="After mobility")
    ax.set_title(GDP_CLASS_SHORT[cls], fontsize=7.5, fontweight="bold")
    ax.set_xticks(TREND_YEARS)
    ax.set_xticklabels([str(y) for y in TREND_YEARS], fontsize=5.5, rotation=45, ha="right")
    ax.set_ylim(0, a_ymax)
    ax.tick_params(axis="y", labelsize=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", lw=0.4, alpha=0.35, linestyle="--", zorder=0)
    if j == 0:
        ax.set_ylabel("Bed occupancy rate (pp)", fontsize=6.5)
    else:
        ax.set_yticklabels([])

# ---- Panel c: Bar plot of relative change using aggregated data ──────────────
row_c_axes = []
year_positions = np.arange(len(TREND_YEARS))

# Calculate global y limits for panel c
all_rel_vals = rel_change_agg_df["relative_change"].dropna()
if len(all_rel_vals) > 0:
    c_ymax = max(abs(all_rel_vals.max()), abs(all_rel_vals.min())) * 1.4
    c_ymax = max(c_ymax, 2)
else:
    c_ymax = 2

for j, cls in enumerate(GDP_CLASS_ORDER):
    ax = fig.add_subplot(gs[2, j*C_COLS:(j+1)*C_COLS])
    row_c_axes.append(ax)
    
    sub_data = rel_change_agg_df[rel_change_agg_df["gdp_class"] == cls]
    
    for i, year in enumerate(TREND_YEARS):
        year_data = sub_data[sub_data["year"] == year]
        if len(year_data) > 0:
            val = year_data["relative_change"].values[0]
        else:
            val = np.nan
        
        if not np.isnan(val):
            bar = ax.bar(i, val, 
                        color=GDP_CLASS_COLORS[cls], width=0.6,
                        edgecolor="#555555", linewidth=0.5, zorder=3)
            
            # Add value label on top/bottom of bar
            offset = abs(val) * 0.05 + 0.3
            va = "bottom" if val >= 0 else "top"
            ax.text(i, val + (offset if val >= 0 else -offset), 
                   f"{val:.1f}%", ha="center", va=va,
                   fontsize=5, color="#333333", fontweight="bold")
    
    ax.axhline(0, color="#888888", lw=0.8, zorder=2)
    ax.set_title(GDP_CLASS_SHORT[cls], fontsize=7.5, fontweight="bold")
    ax.set_xticks(year_positions)
    ax.set_xticklabels([str(y) for y in TREND_YEARS], fontsize=5.5, rotation=45, ha="right")
    ax.set_ylim(-c_ymax, c_ymax)
    ax.tick_params(axis="y", labelsize=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", lw=0.4, alpha=0.35, linestyle="--", zorder=0)
    
    if j == 0:
        ax.set_ylabel("Relative change in occupancy (%)", fontsize=6.5)
    else:
        ax.set_yticklabels([])

# ---- Legends ──────────────────────────────────────────────────────────────────
fig.canvas.draw()

row_a_bottom = min(a.get_position().y0 for a in row_a_axes)
fig.legend(
    handles=[plt.Line2D([0], [0], marker="s", color="w",
                         markerfacecolor=GDP_CLASS_COLORS[c], markersize=7,
                         label=GDP_CLASS_SHORT[c]) for c in GDP_CLASS_ORDER],
    fontsize=6.5, frameon=False, ncol=5, loc="upper center",
    bbox_to_anchor=(0.5, row_a_bottom - 0.035)
)

row_b_bottom = min(a.get_position().y0 for a in row_b_axes)
fig.legend(
    handles=[
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=STATE_COLORS["before"],
                   markersize=5, label="Before mobility"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=STATE_COLORS["after"],
                   markersize=5, label="After mobility"),
    ],
    fontsize=6.5, frameon=False, ncol=2, loc="upper center",
    bbox_to_anchor=(0.5, row_b_bottom - 0.035)
)

row_a_top = max(a.get_position().y1 for a in row_a_axes)
row_b_top = max(a.get_position().y1 for a in row_b_axes)
row_c_top = max(a.get_position().y1 for a in row_c_axes)

fig.text(0.01, row_a_top + 0.015, "a", fontsize=10, fontweight="bold", va="top", ha="left")
fig.text(0.01, row_b_top + 0.015, "b", fontsize=10, fontweight="bold", va="top", ha="left")
fig.text(0.01, row_c_top + 0.015, "c", fontsize=10, fontweight="bold", va="top", ha="left")

fig.savefig(OUTFILE, dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"\n\u2713 Saved \u2192 {OUTFILE}")

✓ Computed occupancy for year 2020 | 350 cities
✓ Computed occupancy for year 2030 | 350 cities
✓ Computed occupancy for year 2040 | 350 cities
✓ Computed occupancy for year 2050 | 350 cities
✓ Computed occupancy for year 2060 | 350 cities

--- Relative change calculated from aggregated data (panel b means) ---

Low:
  2020: before=4.28%, after=3.48%, delta=-0.8039 pp, rel_change=-18.77%
  2030: before=3.87%, after=3.14%, delta=-0.7339 pp, rel_change=-18.97%
  2040: before=3.17%, after=2.54%, delta=-0.6322 pp, rel_change=-19.92%
  2050: before=2.84%, after=2.24%, delta=-0.6081 pp, rel_change=-21.38%
  2060: before=2.01%, after=1.54%, delta=-0.4671 pp, rel_change=-23.29%

Lower middle:
  2020: before=3.15%, after=2.65%, delta=-0.4948 pp, rel_change=-15.73%
  2030: before=3.08%, after=2.65%, delta=-0.4327 pp, rel_change=-14.04%
  2040: before=2.81%, after=2.39%, delta=-0.4215 pp, rel_change=-14.98%
  2050: before=2.63%, after=2.20%, delta=-0.4322 pp, rel_change=-16.42%
  2060: before=2.0